In [81]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import dotenv
from dotenv import load_dotenv
import os
import time 
import requests
from tqdm import tqdm

In [82]:
# use only entries post 2010 JAN 01

df = pd.read_csv("complete.csv", usecols=range(0, 11))
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 88875 entries, 0 to 88874
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              88875 non-null  str    
 1   city                  88679 non-null  str    
 2   state                 81356 non-null  str    
 3   country               76314 non-null  str    
 4   shape                 85757 non-null  str    
 5   duration (seconds)    88873 non-null  object 
 6   duration (hours/min)  85772 non-null  str    
 7   comments              88749 non-null  str    
 8   date posted           88875 non-null  str    
 9   latitude              88875 non-null  str    
 10  longitude             88875 non-null  float64
dtypes: float64(1), object(1), str(9)
memory usage: 7.5+ MB


C:\Users\wslid\AppData\Local\Temp\ipykernel_29296\586527434.py:3: DtypeWarning: Columns (0: duration (seconds)) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("complete.csv", usecols=range(0, 11))


In [87]:
# convert datetime to datetime datatype

df['datetime'] = pd.to_datetime(df['datetime'].astype(str), errors='coerce')

print("Datetime conversion summary:")
print(df['datetime'].describe())
print(f"\nMissing datetimes: {df['datetime'].isna().sum():,}")

# Drop rows that couldn't be parsed 
df = df.dropna(subset=['datetime']).copy()

# post-2010 sightings dataframe to insert weather data into
# Fix: Use a string or proper datetime for comparison
sighting_wx_df = df[df['datetime'] >= '2010-01-01'].copy()
sighting_wx_df['rounded_dt'] = sighting_wx_df['datetime'].dt.round('h')
sighting_wx_df['date_str'] = sighting_wx_df['rounded_dt'].dt.strftime('%Y-%m-%d')
sighting_wx_df['hour_str'] = sighting_wx_df['rounded_dt'].dt.strftime('%H') 

sighting_wx_df.head()



Datetime conversion summary:
count                         87613
mean     2004-04-04 08:00:34.616095
min             1906-11-11 00:00:00
25%             2001-06-25 21:00:00
50%             2006-10-03 23:45:00
75%             2011-05-22 19:31:00
max             2014-05-08 18:45:00
Name: datetime, dtype: object

Missing datetimes: 0


,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str
230,2010-10-10 01:00:00,orchard park,ny,us,light,7200,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,01
231,2010-10-10 02:30:00,harrisburg,pa,us,circle,240,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,02
232,2010-10-10 03:00:00,euclid,oh,us,circle,180,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,03
233,2010-10-10 08:30:00,starr,sc,us,formation,600,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,08
234,2010-10-10 10:45:00,leominster,ma,us,flash,0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11


In [89]:
wx_api_key = 'd1b7077b7ab84b9f9c0160725260405'



# Add the new columns first (with None values)
sighting_wx_df['weather_cloud'] = None
sighting_wx_df['weather_temp_f'] = None
sighting_wx_df['weather_condition'] = None

print(f"Starting weather fetch for {len(sighting_wx_df):,} sightings...")


for idx, row in tqdm(sighting_wx_df.iterrows(), total=len(sighting_wx_df)):
    try:
        lat = str(row['latitude']).strip()
        lon = str(row['longitude']).strip()
        date = str(row['date_str']).strip()
        hour = str(row['hour_str']).strip()
        
        baseurl = "http://api.weatherapi.com/v1/"
        historyurl = (
            f"{baseurl}history.json"
            f"?key={wx_api_key}"
            f"&q={lat},{lon}"          
            f"&dt={date}"
            f"&hour={hour}"            
        )
        
        response = requests.get(historyurl, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        hour_data = data['forecast']['forecastday'][0]['hour'][0]
        
        # Update the dataframe row by row
        sighting_wx_df.at[idx, 'weather_cloud']     = hour_data.get('cloud')
        sighting_wx_df.at[idx, 'weather_temp_f']    = hour_data.get('temp_f')
        sighting_wx_df.at[idx, 'weather_condition'] = hour_data.get('condition', {}).get('text')
        
        #time.sleep(0.1)        
        
    except Exception:
        pass

    if (idx + 1) % 500 == 0:
        checkpoint_file = f"sightings_weather_checkpoint_{idx+1}.csv"
        sighting_wx_df.to_csv(checkpoint_file, index=False)
        print(f"\n Checkpoint saved at row {idx+1} → {checkpoint_file}")


print(sighting_wx_df.head(10))

# Save the result
sighting_wx_df.to_csv("sightings_with_weather.csv", index=False)

Starting weather fetch for 28,022 sightings...


  0%|          | 121/28022 [00:30<2:53:33,  2.68it/s]


 Checkpoint saved at row 500 → sightings_weather_checkpoint_500.csv


  1%|          | 150/28022 [00:37<2:38:39,  2.93it/s]


 Checkpoint saved at row 1000 → sightings_weather_checkpoint_1000.csv


...


 Checkpoint saved at row 85500 → sightings_weather_checkpoint_85500.csv


 96%|█████████▋| 27016/28022 [1:45:46<04:02,  4.14it/s]


 Checkpoint saved at row 86000 → sightings_weather_checkpoint_86000.csv


100%|██████████| 28022/28022 [1:49:15<00:00,  4.27it/s]

               datetime                   city state country      shape  \
230 2010-10-10 01:00:00           orchard park    ny      us      light   
231 2010-10-10 02:30:00             harrisburg    pa      us     circle   
232 2010-10-10 03:00:00                 euclid    oh      us     circle   
233 2010-10-10 08:30:00                  starr    sc      us  formation   
234 2010-10-10 10:45:00             leominster    ma      us      flash   
235 2010-10-10 12:00:00              greenwich    ct      us      light   
236 2010-10-10 13:00:00       north charleston    sc      us      light   
237 2010-10-10 15:00:00  san francisco airport    ca     NaN   triangle   
238 2010-10-10 16:30:00            springfield    va      us      cigar   
239 2010-10-10 17:10:00             bridgeport    ct      us      light   

    duration (seconds) duration (hours/min)  \
230               7200          a few hours   
231                240           4  minutes   
232                180           